<a href="https://colab.research.google.com/github/Mrshayan07/ANN-House-Price-Prediction/blob/main/ANN_House_price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile
zip = zipfile.ZipFile("/content/zameen-updated.csv.zip",'r')
zip.extractall("/content")
zip.close()

In [ ]:
folder_path = '/content/zameen-updated.csv'

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("zameen-updated.csv")

# Remove invalid prices
df = df[df['price'] > 0]

# Remove extreme outliers
df = df[df['baths'] <= 20]
df = df[df['bedrooms'] <= 20]
df = df[df['Area Size'] <= 100]

# Remove duplicates
df = df.drop_duplicates()

In [ ]:
drop_cols = [
    'property_id',
    'location_id',
    'page_url',
    'agency',
    'agent',
    'date_added'
]

df.drop(columns=drop_cols, inplace=True, errors='ignore')

In [ ]:
df['lat_long_interaction'] = (
    df['latitude'] * df['longitude']
)

In [ ]:
df['price'] = np.log1p(df['price'])

In [ ]:
df['price']

,price
0,16.118096
1,15.747032
2,16.618871
3,17.588272
4,15.761421
...,...
168441,17.092655
168442,16.341239
168443,17.111347
168444,16.213406


In [ ]:
X = df.drop('price', axis=1)
y = df['price']

In [ ]:
categorical_cols = X.select_dtypes(
    include=['object']
).columns

numerical_cols = X.select_dtypes(
    exclude=['object']
).columns

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

In [ ]:
numeric_transformer

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization

In [ ]:
model = Sequential()

model.add(Dense(
    512,
    activation='relu',
    input_shape=(X_train.shape[1],)
))

model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(
    256,
    activation='relu'
))

model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(
    128,
    activation='relu'
))

model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Dense(
    64,
    activation='relu'
))

model.add(Dense(1))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
from tensorflow.keras.optimizers import Adam

optimizer = Adam(
    learning_rate=0.001
)

model.compile(
    optimizer=optimizer,
    loss='huber',
    metrics=['mae']
)

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

In [ ]:
early_stop = EarlyStopping(
    patience=20,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    factor=0.5,
    patience=5
)

checkpoint = ModelCheckpoint(
    "best_house_model.keras",
    save_best_only=True
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=300,
    batch_size=128,
    callbacks=[
        early_stop,
        reduce_lr,
        checkpoint
    ]
)

Epoch 1/300
843/843 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.9180 - mae: 1.3110 - val_loss: 0.1157 - val_mae: 0.3743 - learning_rate: 0.0010
Epoch 2/300
843/843 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.3750 - mae: 0.7344 - val_loss: 0.0841 - val_mae: 0.3116 - learning_rate: 0.0010
Epoch 3/300
843/843 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.3124 - mae: 0.6568 - val_loss: 0.1328 - val_mae: 0.4054 - learning_rate: 0.0010
Epoch 4/300
843/843 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.2504 - mae: 0.5763 - val_loss: 0.1254 - val_mae: 0.3984 - learning_rate: 0.0010
Epoch 5/300
843/843 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.1937 - mae: 0.4959 - val_loss: 0.1290 - val_mae: 0.3632 - learning_rate: 0.0010
Epoch 6/300
843/843 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.1398 - mae: 0.4121 - val_loss: 0.0638 - val_mae: 0.2609 - learning_rate: 0.0010
Epoch 7/300
843/843 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.1043 - mae: 0.3496 - val_loss: 0.0864 - val_mae: 0.3269 - learning_rate: 0.00

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

preds = model.predict(X_test)

preds = np.expm1(preds)
actual = np.expm1(y_test)

mae = mean_absolute_error(actual, preds)
rmse = np.sqrt(
    mean_squared_error(actual, preds)
)
r2 = r2_score(actual, preds)

print(mae)
print(rmse)
print(r2)

1053/1053 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step
3693959.698433831
20330239.687009756
0.6488855776619493


In [ ]:
import joblib

model.save("house_price_model.keras")

joblib.dump(
    preprocessor,
    "preprocessor.pkl"
)

['preprocessor.pkl']

In [ ]:
from tensorflow.keras.models import load_model
import joblib
import pandas as pd
import numpy as np

model = load_model(
    "house_price_model.keras"
)

preprocessor = joblib.load(
    "preprocessor.pkl"
)

In [ ]:
import pandas as pd
import numpy as np

def predict_house_price(
    property_type,
    location,
    city,
    province_name,
    latitude,
    longitude,
    baths,
    area,
    purpose,
    bedrooms,
    area_type,
    area_size,
    area_category
):

    sample = pd.DataFrame([{
        "property_type": property_type,
        "location": location,
        "city": city,
        "province_name": province_name,
        "latitude": latitude,
        "longitude": longitude,
        "baths": baths,
        "area": area,
        "purpose": purpose,
        "bedrooms": bedrooms,
        "Area Type": area_type,
        "Area Size": area_size,
        "Area Category": area_category
    }])

    # Same feature engineering as training
    sample["lat_long_interaction"] = (
        sample["latitude"] * sample["longitude"]
    )

    # IMPORTANT: column order same as training
    sample = sample[
        [
            'property_type',
            'location',
            'city',
            'province_name',
            'latitude',
            'longitude',
            'baths',
            'area',
            'purpose',
            'bedrooms',
            'Area Type',
            'Area Size',
            'Area Category',
            'lat_long_interaction'
        ]
    ]

    X = preprocessor.transform(sample)

    pred_log = model.predict(X, verbose=0)

    pred_price = np.expm1(pred_log)[0][0]

    return round(pred_price)

In [ ]:
price = predict_house_price(
    property_type="House",
    location="DHA Phase 6",
    city="Lahore",
    province_name="Punjab",
    latitude=31.4697,
    longitude=74.4123,
    baths=5,
    area=10,
    purpose="For Sale",
    bedrooms=5,
    area_type="Kanal",
    area_size=10,
    area_category="10 Kanal"
)

print(f"Predicted Price: PKR {price:,}")

Predicted Price: PKR 47,201,272


In [ ]:
row = df[
    (df['property_type'] == 'House') &
    (df['city'] == 'Lahore') &
    (df['province_name'] == 'Punjab') &
    (df['baths'] == 5) &
    (df['bedrooms'] == 5) &
    (df['purpose'] == 'For Sale')
]

print(row.head())
print("Rows Found:", len(row))

    property_type      price          location    city province_name  \
48          House  17.504390       Multan Road  Lahore        Punjab   
78          House  17.867296       EME Society  Lahore        Punjab   
87          House  16.300417  Al-Raheem Garden  Lahore        Punjab   
92          House  16.648724      Paragon City  Lahore        Punjab   
300         House  16.929026            Askari  Lahore        Punjab   

      latitude  longitude  baths      area   purpose  bedrooms Area Type  \
48   31.431593  74.179980      5   1 Kanal  For Sale         5     Kanal   
78   31.439978  74.209685      5   1 Kanal  For Sale         5     Kanal   
87   31.597234  74.474513      5   6 Marla  For Sale         5     Marla   
92   31.530739  74.456184      5  10 Marla  For Sale         5     Marla   
300  31.537458  74.413323      5  10 Marla  For Sale         5     Marla   

     Area Size Area Category  lat_long_interaction  
48         1.0     1-5 Kanal           2331.594940  
78  

In [ ]:
row = df[
    (df['property_type'] == 'House') &
    (df['location'].str.contains('DHA Phase 6', case=False, na=False)) &
    (df['city'] == 'Lahore')
]

print(row[['price',
           'location',
           'city',
           'baths',
           'bedrooms',
           'Area Size']].head(10))

Empty DataFrame
Columns: [price, location, city, baths, bedrooms, Area Size]
Index: []


In [ ]:
sample_lat = 31.4697
sample_lon = 74.4123

df['distance'] = (
    (df['latitude'] - sample_lat)**2 +
    (df['longitude'] - sample_lon)**2
)

closest = df.sort_values('distance').head(1)

print(closest.T)

                                    13416
property_type                       House
price                           16.166886
location              UBL Housing Society
city                               Lahore
province_name                      Punjab
latitude                        31.473125
longitude                       74.413574
baths                                   4
area                            4.5 Marla
purpose                          For Sale
bedrooms                                4
Area Type                           Marla
Area Size                             4.5
Area Category                   0-5 Marla
lat_long_interaction          2342.027716
distance                         0.000013


In [ ]:
price = predict_house_price(
    property_type="House",
    location="Soan Garden",
    city="Islamabad",
    province_name="Islamabad Capital",
    latitude=33.564427,
    longitude=73.157101,
    baths=0,
    area="8 Marla",
    purpose="For Sale",
    bedrooms=0,
    area_type="Marla",
    area_size=8.0,
    area_category="5-10 Marla"
)

print(f"Predicted Price: PKR {price:,}")

Predicted Price: PKR 13,014,424


In [ ]:
df.sample(1)

,property_type,price,location,city,province_name,latitude,longitude,baths,area,purpose,bedrooms,Area Type,Area Size,Area Category,lat_long_interaction,distance
25985,House,16.588099,Soan Garden,Islamabad,Islamabad Capital,33.564427,73.157101,0,8 Marla,For Sale,0,Marla,8.0,5-10 Marla,2455.476176,5.963406


In [ ]:
df['price'] = np.log1p(df['price'])

In [ ]:
actual_price = np.expm1(16.588099)

print(actual_price)

15999994.516735738


In [ ]:
actual_price = np.expm1(closest['price'].iloc[0])

predicted_price = predict_house_price(
    property_type="House",
    location="Soan Garden",
    city="Islamabad",
    province_name="Islamabad Capital",
    latitude=33.564427,
    longitude=73.157101,
    baths=0,
    area="8 Marla",
    purpose="For Sale",
    bedrooms=0,
    area_type="Marla",
    area_size=8.0,
    area_category="5-10 Marla"
)

print(f"Actual Price: PKR {actual_price:,.0f}")
print(f"Predicted Price: PKR {predicted_price:,.0f}")
print(f"Absolute Error: PKR {abs(actual_price - predicted_price):,.0f}")

error_percent = abs(actual_price - predicted_price) / actual_price * 100

print(f"Percentage Error: {error_percent:.2f}%")

Actual Price: PKR 10,500,000
Predicted Price: PKR 13,014,424
Absolute Error: PKR 2,514,424
Percentage Error: 23.95%
